In [18]:
from pathlib import Path

BASE_DIR = Path(r"D:\AI_IT_HELPDESK")

KB_DIR = BASE_DIR / "knowledge_base"
DOC_DIR = KB_DIR / "documents"
VECTOR_DIR = KB_DIR / "vector_store"

RESULT_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"

for folder in [
    KB_DIR, DOC_DIR, VECTOR_DIR,
    RESULT_DIR, MODEL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("All project paths configured successfully!")

All project paths configured successfully!


In [ ]:
# Project configuration

In [1]:
from pathlib import Path

BASE_DIR = Path(r"D:\AI_IT_HELPDESK")

KB_DIR = BASE_DIR / "knowledge_base"
DOC_DIR = KB_DIR / "documents"
VECTOR_DIR = KB_DIR / "vector_store"

RESULT_DIR = BASE_DIR / "results"

DOC_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", BASE_DIR)
print("Knowledge Base:", KB_DIR)
print("Documents:", DOC_DIR)
print("Vector Database:", VECTOR_DIR)

assert BASE_DIR.exists(), "Project folder not found!"

print("\nRAG project directories ready!")

Project: D:\AI_IT_HELPDESK
Knowledge Base: D:\AI_IT_HELPDESK\knowledge_base
Documents: D:\AI_IT_HELPDESK\knowledge_base\documents
Vector Database: D:\AI_IT_HELPDESK\knowledge_base\vector_store

RAG project directories ready!


In [2]:
%pip install -q sentence-transformers chromadb
%pip install -q pypdf rank-bm25

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
import uuid
import pandas as pd
import numpy as np
import chromadb

from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from pathlib import Path

print("RAG libraries imported successfully!")
print("ChromaDB version:", chromadb.__version__)

RAG libraries imported successfully!
ChromaDB version: 1.5.9


In [ ]:
# Knowledge base creation

In [2]:
knowledge_articles = [

    # ACCOUNT AND PASSWORD
    {
        "id": "KB001",
        "category": "Account & Password",
        "title": "Windows Login Failure",
        "content": """
        Problem: User cannot log into Windows.

        Troubleshooting:
        1. Verify the username.
        2. Check keyboard layout and Caps Lock.
        3. Record the displayed error message.
        4. Check whether the account is locked or disabled.
        5. Contact an authorised administrator if required.

        Safety: Never request or store the user's password.
        """,
        "source": "Demo IT Support Knowledge Base"
    },
    {
        "id": "KB002",
        "category": "Account & Password",
        "title": "Password Reset",
        "content": """
        Problem: User forgot the Windows password.

        Troubleshooting:
        1. Verify the user's identity.
        2. Follow the organisation's approved password reset procedure.
        3. Confirm successful authentication.
        4. Never share passwords through chat or email.

        Safety: Password reset requires authorised procedures.
        """,
        "source": "Demo IT Support Knowledge Base"
    },
    {
        "id": "KB003",
        "category": "Account & Password",
        "title": "Account Locked",
        "content": """
        Problem: Windows account is locked.

        Troubleshooting:
        1. Record the account lockout error.
        2. Check whether repeated login failures occurred.
        3. Verify account status with an administrator.
        4. Follow the approved account unlock process.

        Safety: Do not bypass account security controls.
        """,
        "source": "Demo IT Support Knowledge Base"
    },

    # NETWORK AND VPN
    {
        "id": "KB004",
        "category": "Network & VPN",
        "title": "Internet Connection Failure",
        "content": """
        Problem: Computer cannot access the internet.

        Troubleshooting:
        1. Check network cable or WiFi connection.
        2. Check network adapter status.
        3. Run ipconfig to inspect IP configuration.
        4. Test connectivity to the default gateway.
        5. Check DNS resolution.
        6. Contact network support if the issue continues.

        Safety: Use read-only diagnostic commands first.
        """,
        "source": "Demo IT Support Knowledge Base"
    },
    {
        "id": "KB005",
        "category": "Network & VPN",
        "title": "VPN Connection Failure",
        "content": """
        Problem: User cannot connect to the company VPN.

        Troubleshooting:
        1. Confirm internet connectivity.
        2. Check the VPN client status.
        3. Verify the VPN server address.
        4. Check authentication and MFA errors.
        5. Record the error message.
        6. Escalate to the network administrator.

        Safety: Never request OTP codes or bypass MFA.
        """,
        "source": "Demo IT Support Knowledge Base"
    },
    {
        "id": "KB006",
        "category": "Network & VPN",
        "title": "DNS Resolution Failure",
        "content": """
        Problem: DNS server is not responding.

        Troubleshooting:
        1. Check IP configuration.
        2. Verify the configured DNS server.
        3. Use nslookup to test name resolution.
        4. Compare results with a known working system.
        5. Escalate persistent DNS issues.

        Safety: Do not change DNS settings without authorisation.
        """,
        "source": "Demo IT Support Knowledge Base"
    },

    # PRINTER
    {
        "id": "KB007",
        "category": "Printer",
        "title": "Printer Offline",
        "content": """
        Problem: Network printer shows offline.

        Troubleshooting:
        1. Check printer power and display.
        2. Check network connection.
        3. Verify the printer IP address.
        4. Check the Windows printer status.
        5. Inspect the print queue.
        6. Contact printer support if unresolved.

        Safety: Do not install unverified printer drivers.
        """,
        "source": "Demo IT Support Knowledge Base"
    },
    {
        "id": "KB008",
        "category": "Printer",
        "title": "Print Queue Stuck",
        "content": """
        Problem: Documents remain in the print queue.

        Troubleshooting:
        1. Open the printer queue.
        2. Identify the stuck document.
        3. Record any displayed error.
        4. Follow the approved queue recovery process.
        5. Test printing with a small document.

        Safety: Obtain approval before stopping print services.
        """,
        "source": "Demo IT Support Knowledge Base"
    },
    {
        "id": "KB009",
        "category": "Printer",
        "title": "Printer Driver Error",
        "content": """
        Problem: Printer driver is not functioning correctly.

        Troubleshooting:
        1. Identify the printer manufacturer and model.
        2. Check the installed driver version.
        3. Review Windows device status.
        4. Obtain an approved driver.
        5. Escalate installation issues if permissions are required.

        Safety: Do not install drivers from untrusted sources.
        """,
        "source": "Demo IT Support Knowledge Base"
    },

    # SOFTWARE
    {
        "id": "KB010",
        "category": "Software",
        "title": "Application Not Opening",
        "content": """
        Problem: Application fails to open.

        Troubleshooting:
        1. Record the application name.
        2. Capture the error message.
        3. Check application compatibility.
        4. Review application logs.
        5. Check whether updates are available.
        6. Escalate persistent errors.

        Safety: Do not disable endpoint security.
        """,
        "source": "Demo IT Support Knowledge Base"
    },
    {
        "id": "KB011",
        "category": "Software",
        "title": "Software Installation Failure",
        "content": """
        Problem: Software installation fails.

        Troubleshooting:
        1. Check the installer file.
        2. Verify system requirements.
        3. Review installation error messages.
        4. Check user installation permissions.
        5. Request authorised software deployment if necessary.

        Safety: Only use approved software packages.
        """,
        "source": "Demo IT Support Knowledge Base"
    },
    {
        "id": "KB012",
        "category": "Software",
        "title": "Application Crashing",
        "content": """
        Problem: Application crashes repeatedly.

        Troubleshooting:
        1. Record when the application crashes.
        2. Identify the application version.
        3. Check available system resources.
        4. Review application error logs.
        5. Check known software updates.
        6. Escalate if crashes continue.

        Safety: Preserve logs before changing configurations.
        """,
        "source": "Demo IT Support Knowledge Base"
    },

    # SYSTEM HEALTH
    {
        "id": "KB013",
        "category": "System Health",
        "title": "High CPU Usage",
        "content": """
        Problem: Computer has unusually high CPU usage.

        Troubleshooting:
        1. Open Task Manager.
        2. Inspect CPU utilisation.
        3. Identify processes using high CPU.
        4. Record application names.
        5. Review system logs.
        6. Escalate suspicious processes.

        Safety: Do not terminate unknown security processes.
        """,
        "source": "Demo IT Support Knowledge Base"
    },
    {
        "id": "KB014",
        "category": "System Health",
        "title": "High Memory Usage",
        "content": """
        Problem: Computer has high RAM usage.

        Troubleshooting:
        1. Open Task Manager.
        2. Check memory utilisation.
        3. Identify high memory applications.
        4. Record available memory.
        5. Review resource-intensive processes.
        6. Escalate persistent performance problems.

        Safety: Do not terminate critical system processes.
        """,
        "source": "Demo IT Support Knowledge Base"
    },
    {
        "id": "KB015",
        "category": "System Health",
        "title": "Low Disk Space",
        "content": """
        Problem: Computer reports insufficient disk space.

        Troubleshooting:
        1. Check available disk capacity.
        2. Identify large folders.
        3. Review temporary file usage.
        4. Follow approved disk cleanup procedures.
        5. Escalate if storage expansion is required.

        Safety: Do not delete business records or user documents.
        """,
        "source": "Demo IT Support Knowledge Base"
    }
]

kb_df = pd.DataFrame(knowledge_articles)

print("Knowledge base created!")
print("Total articles:", len(kb_df))

display(kb_df[["id", "category", "title", "source"]])


Knowledge base created!
Total articles: 15


,id,category,title,source
0,KB001,Account & Password,Windows Login Failure,Demo IT Support Knowledge Base
1,KB002,Account & Password,Password Reset,Demo IT Support Knowledge Base
2,KB003,Account & Password,Account Locked,Demo IT Support Knowledge Base
3,KB004,Network & VPN,Internet Connection Failure,Demo IT Support Knowledge Base
4,KB005,Network & VPN,VPN Connection Failure,Demo IT Support Knowledge Base
5,KB006,Network & VPN,DNS Resolution Failure,Demo IT Support Knowledge Base
6,KB007,Printer,Printer Offline,Demo IT Support Knowledge Base
7,KB008,Printer,Print Queue Stuck,Demo IT Support Knowledge Base
8,KB009,Printer,Printer Driver Error,Demo IT Support Knowledge Base
9,KB010,Software,Application Not Opening,Demo IT Support Knowledge Base


In [ ]:
# Save knowledge documents

In [4]:

from pathlib import Path

# Project directory
BASE_DIR = Path(r"D:\AI_IT_HELPDESK")

# Knowledge base directory
KB_DIR = BASE_DIR / "knowledge_base"

# Documents directory
DOC_DIR = KB_DIR / "documents"

# Vector database directory
VECTOR_DIR = KB_DIR / "vector_store"

# Create folders if not available
KB_DIR.mkdir(parents=True, exist_ok=True)
DOC_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

# CSV file path
KB_CSV = KB_DIR / "it_knowledge_base.csv"

print("KB_DIR:", KB_DIR)
print("KB_CSV:", KB_CSV)
print("Configuration successful!")


KB_DIR: D:\AI_IT_HELPDESK\knowledge_base
KB_CSV: D:\AI_IT_HELPDESK\knowledge_base\it_knowledge_base.csv
Configuration successful!


In [5]:
kb_df.to_csv(
    KB_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Knowledge base saved successfully!")
print("Location:", KB_CSV)
print("Total articles:", len(kb_df))


Knowledge base saved successfully!
Location: D:\AI_IT_HELPDESK\knowledge_base\it_knowledge_base.csv
Total articles: 15


In [ ]:
# Text chunking

In [6]:
def create_chunks(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks


documents = []
metadatas = []
ids = []

for article in knowledge_articles:

    full_text = (
        f"Title: {article['title']}\n"
        f"Category: {article['category']}\n"
        f"{article['content']}"
    )

    chunks = create_chunks(full_text)

    for index, chunk in enumerate(chunks):

        documents.append(chunk)

        metadatas.append({
            "article_id": article["id"],
            "category": article["category"],
            "title": article["title"],
            "source": article["source"],
            "chunk_number": index
        })

        ids.append(f"{article['id']}_chunk_{index}")

print("Total chunks:", len(documents))
print("\nSample chunk:")
print(documents[0])


Total chunks: 15

Sample chunk:
Title: Windows Login Failure Category: Account & Password Problem: User cannot log into Windows. Troubleshooting: 1. Verify the username. 2. Check keyboard layout and Caps Lock. 3. Record the displayed error message. 4. Check whether the account is locked or disabled. 5. Contact an authorised administrator if required. Safety: Never request or store the user's password.


In [ ]:
# Load embedding model

In [7]:

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cpu"
)

print("Embedding model loaded!")
print("Model:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:",
      embedding_model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Nimesh\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nimesh\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!
Model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


C:\Users\Nimesh\AppData\Local\Temp\ipykernel_31132\2003097983.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension())


In [ ]:
# Generate embeddings

In [8]:
embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings generated successfully!")
print("Embedding shape:", embeddings.shape)
print("Total document vectors:", len(embeddings))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully!
Embedding shape: (15, 384)
Total document vectors: 15


In [ ]:
# ChromaDB setup

In [9]:
chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_DIR)
)

collection = chroma_client.get_or_create_collection(
    name="it_helpdesk_knowledge",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB initialized!")
print("Collection:", collection.name)

ChromaDB initialized!
Collection: it_helpdesk_knowledge


In [ ]:
# Insert knowledge vectors

In [10]:

if collection.count() > 0:
    collection.delete(ids=collection.get()["ids"])

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Knowledge stored successfully!")
print("Total stored vectors:", collection.count())

Knowledge stored successfully!
Total stored vectors: 15


In [ ]:
# Retrieval function

In [11]:
def retrieve_knowledge(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=min(top_k, collection.count()),
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    retrieved = []

    for i in range(len(results["ids"][0])):

        retrieved.append({
            "document_id": results["ids"][0][i],
            "content": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return retrieved


print("Semantic retrieval function created!")


Semantic retrieval function created!


In [ ]:
# Test RAG search

In [12]:
test_query = "My office printer is offline and cannot print documents"

search_results = retrieve_knowledge(
    test_query,
    top_k=3
)

print("USER QUERY:")
print(test_query)

print("\nRETRIEVED DOCUMENTS")
print("=" * 60)

for rank, result in enumerate(search_results, start=1):

    print("\nResult:", rank)
    print("Article:", result["metadata"]["article_id"])
    print("Title:", result["metadata"]["title"])
    print("Category:", result["metadata"]["category"])
    print("Cosine distance:", round(result["distance"], 4))
    print("Content:", result["content"])


USER QUERY:
My office printer is offline and cannot print documents

RETRIEVED DOCUMENTS

Result: 1
Article: KB007
Title: Printer Offline
Category: Printer
Cosine distance: 0.2593
Content: Title: Printer Offline Category: Printer Problem: Network printer shows offline. Troubleshooting: 1. Check printer power and display. 2. Check network connection. 3. Verify the printer IP address. 4. Check the Windows printer status. 5. Inspect the print queue. 6. Contact printer support if unresolved. Safety: Do not install unverified printer drivers.

Result: 2
Article: KB008
Title: Print Queue Stuck
Category: Printer
Cosine distance: 0.4696
Content: Title: Print Queue Stuck Category: Printer Problem: Documents remain in the print queue. Troubleshooting: 1. Open the printer queue. 2. Identify the stuck document. 3. Record any displayed error. 4. Follow the approved queue recovery process. 5. Test printing with a small document. Safety: Obtain approval before stopping print services.

Result: 3
Arti

In [ ]:
# Grounded answer generator

In [13]:

def generate_grounded_answer(query, top_k=3):

    results = retrieve_knowledge(query, top_k)

    if not results:
        return {
            "answer": "No relevant knowledge was found. "
                      "Please escalate to IT support.",
            "sources": []
        }

    best = results[0]

    # Conservative retrieval threshold.
    # This is a prototype setting and requires calibration.
    if best["distance"] > 0.65:
        return {
            "answer": (
                "I could not find sufficiently relevant "
                "troubleshooting evidence. "
                "Please escalate to an IT analyst."
            ),
            "sources": []
        }

    answer = (
        "Based on the retrieved knowledge article:\n\n"
        + best["content"]
        + "\n\nPlease follow only authorised procedures. "
        "If the issue remains unresolved, contact IT support."
    )

    sources = [
        {
            "article_id": item["metadata"]["article_id"],
            "title": item["metadata"]["title"],
            "source": item["metadata"]["source"]
        }
        for item in results
    ]

    return {
        "answer": answer,
        "sources": sources
    }


response = generate_grounded_answer(
    "My printer is offline"
)

print("AI TROUBLESHOOTING RESPONSE")
print("=" * 50)
print(response["answer"])

print("\nSOURCES:")
for source in response["sources"]:
    print(source)


AI TROUBLESHOOTING RESPONSE
Based on the retrieved knowledge article:

Title: Printer Offline Category: Printer Problem: Network printer shows offline. Troubleshooting: 1. Check printer power and display. 2. Check network connection. 3. Verify the printer IP address. 4. Check the Windows printer status. 5. Inspect the print queue. 6. Contact printer support if unresolved. Safety: Do not install unverified printer drivers.

Please follow only authorised procedures. If the issue remains unresolved, contact IT support.

SOURCES:
{'article_id': 'KB007', 'title': 'Printer Offline', 'source': 'Demo IT Support Knowledge Base'}
{'article_id': 'KB009', 'title': 'Printer Driver Error', 'source': 'Demo IT Support Knowledge Base'}
{'article_id': 'KB008', 'title': 'Print Queue Stuck', 'source': 'Demo IT Support Knowledge Base'}


In [ ]:
# Retrieval evaluation

In [14]:
evaluation_queries = [
    {
        "query": "My printer is offline",
        "expected_id": "KB007"
    },
    {
        "query": "I cannot connect to office VPN",
        "expected_id": "KB005"
    },
    {
        "query": "I forgot my Windows password",
        "expected_id": "KB002"
    },
    {
        "query": "My application keeps crashing",
        "expected_id": "KB012"
    },
    {
        "query": "My computer has high CPU usage",
        "expected_id": "KB013"
    }
]

evaluation_results = []

for item in evaluation_queries:

    results = retrieve_knowledge(
        item["query"],
        top_k=3
    )

    retrieved_ids = [
        result["metadata"]["article_id"]
        for result in results
    ]

    relevant_found = (
        item["expected_id"] in retrieved_ids
    )

    evaluation_results.append({
        "Query": item["query"],
        "Expected Article": item["expected_id"],
        "Retrieved Articles": ", ".join(retrieved_ids),
        "Relevant Found": relevant_found
    })

eval_df = pd.DataFrame(evaluation_results)

display(eval_df)

recall_at_3 = eval_df["Relevant Found"].mean()

print("Recall@3:", round(recall_at_3, 4))
print("Recall@3 Percentage:", round(recall_at_3 * 100, 2), "%")

,Query,Expected Article,Retrieved Articles,Relevant Found
0,My printer is offline,KB007,"KB007, KB009, KB008",True
1,I cannot connect to office VPN,KB005,"KB005, KB004, KB006",True
2,I forgot my Windows password,KB002,"KB002, KB001, KB003",True
3,My application keeps crashing,KB012,"KB012, KB010, KB014",True
4,My computer has high CPU usage,KB013,"KB013, KB014, KB015",True


Recall@3: 1.0
Recall@3 Percentage: 100.0 %


In [ ]:
# Export evaluation RAG

In [16]:
from pathlib import Path

# Project directory
BASE_DIR = Path(r"D:\AI_IT_HELPDESK")

# Results directory
RESULT_DIR = BASE_DIR / "results"

# Create directory if not exists
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Results Directory:", RESULT_DIR)
print("Directory Ready!")


Results Directory: D:\AI_IT_HELPDESK\results
Directory Ready!


In [19]:
EVALUATION_FILE = RESULT_DIR / "rag_retrieval_evaluation.csv"

eval_df.to_csv(
    EVALUATION_FILE,
    index=False,
    encoding="utf-8-sig"
)

summary = pd.DataFrame([{
    "Total Queries": len(eval_df),
    "Relevant Retrieved": int(
        eval_df["Relevant Found"].sum()
    ),
    "Recall_at_3": recall_at_3
}])

summary.to_csv(
    RESULT_DIR / "rag_evaluation_summary.csv",
    index=False
)

display(summary)

print("RAG evaluation saved successfully!")

,Total Queries,Relevant Retrieved,Recall_at_3
0,5,5,1.0


RAG evaluation saved successfully!


In [ ]:
# Interactive troubleshooting test

In [20]:
def ask_helpdesk(question):

    result = generate_grounded_answer(question)

    print("\nUSER:")
    print(question)

    print("\nAI HELP DESK:")
    print(result["answer"])

    print("\nREFERENCE ARTICLES:")

    if result["sources"]:
        for source in result["sources"]:
            print(
                source["article_id"],
                "-",
                source["title"]
            )
    else:
        print("No verified source retrieved.")

    print("\n" + "-" * 55)


ask_helpdesk("My office printer is offline")

ask_helpdesk("I cannot connect to company VPN")

ask_helpdesk("My computer has high RAM usage")

ask_helpdesk("I forgot my Windows password")



USER:
My office printer is offline

AI HELP DESK:
Based on the retrieved knowledge article:

Title: Printer Offline Category: Printer Problem: Network printer shows offline. Troubleshooting: 1. Check printer power and display. 2. Check network connection. 3. Verify the printer IP address. 4. Check the Windows printer status. 5. Inspect the print queue. 6. Contact printer support if unresolved. Safety: Do not install unverified printer drivers.

Please follow only authorised procedures. If the issue remains unresolved, contact IT support.

REFERENCE ARTICLES:
KB007 - Printer Offline
KB008 - Print Queue Stuck
KB009 - Printer Driver Error

-------------------------------------------------------

USER:
I cannot connect to company VPN

AI HELP DESK:
Based on the retrieved knowledge article:

Title: VPN Connection Failure Category: Network & VPN Problem: User cannot connect to the company VPN. Troubleshooting: 1. Confirm internet connectivity. 2. Check the VPN client status. 3. Verify the V

In [ ]:
# Final system check

In [23]:
print("=" * 50)
print("AI IT HELPDESK - RAG SYSTEM REPORT")
print("=" * 50)

print("Knowledge Articles:", len(knowledge_articles))
print("Document Chunks:", len(documents))
print("Embedding Model:", EMBEDDING_MODEL_NAME)
print("Embedding Dimensions:",
      embedding_model.get_sentence_embedding_dimension())

print("ChromaDB Collection:", collection.name)
print("Stored Vectors:", collection.count())

print("Recall@3:", round(recall_at_3, 4))

print("Vector Database Path:", VECTOR_DIR)

print("\nRAG KNOWLEDGE BASE SETUP COMPLETED!")


AI IT HELPDESK - RAG SYSTEM REPORT
Knowledge Articles: 15
Document Chunks: 15
Embedding Model: sentence-transformers/all-MiniLM-L6-v2
Embedding Dimensions: 384
ChromaDB Collection: it_helpdesk_knowledge
Stored Vectors: 15
Recall@3: 1.0
Vector Database Path: D:\AI_IT_HELPDESK\knowledge_base\vector_store

RAG KNOWLEDGE BASE SETUP COMPLETED!


C:\Users\Nimesh\AppData\Local\Temp\ipykernel_31132\3444485918.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension())
